<a href="https://colab.research.google.com/github/starryubin/Gauri_INFO4670_Fall2026/blob/main/Week5_INFO4670_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [1]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [2]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()

**Your justification (1–2 sentences):

I would use central tendency (imputing with the median) or class mean to handle the missing study hours. Since study hours vary by student but typically cluster around an average, replacing missing entries with the median avoids skewing the data with extreme outliers while maintaining a realistic representation of student workload.

In [3]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [4]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups

housing_map = {
    'on-campus': 'On-Campus',
    'off-campus': 'Off-Campus',
    'off campus': 'Off-Campus',
    'with family': 'With Family'
}

students["housing_clean"] = students["housing"].str.strip().str.lower().map(housing_map)

In [5]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [6]:
# TODO
impossible_ages = sorted(students[(students["age"] < 15) | (students["age"] > 100)]["age"].unique())
n_neg_work = int((students["work_hours_per_week"] < 0).sum())

In [7]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [8]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset=["student_id"])

**Why not de-dupe enroll / activity? (1 sentence):

You shouldn't de-duplicate enroll or activity by student ID because students have multiple rows representing different course enrollments and weekly activity logs across time.

In [9]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [10]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
students_dedup["student_id_num"] = students_dedup["student_id"].str.replace("NU-", "").astype(int)
activity_summary = activity.groupby("student_id", as_index=False)[["lms_clicks", "minutes_active"]].sum()
analysis = students_dedup.merge(activity_summary, on="student_id", how="left")

/tmp/ipykernel_14269/4148458929.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  students_dedup["student_id_num"] = students_dedup["student_id"].str.replace("NU-", "").astype(int)


In [11]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [12]:
# TODO
set_of_keys = set(students_dedup["student_id_num"])
matched = int(enroll["sid"].isin(set_of_keys).sum())

In [13]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [14]:
# TODO
dates_parsed = pd.to_datetime(students["enrollment_date"], format="mixed", errors="coerce")

In [15]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [16]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
study = analysis["study_hours_reported"]
analysis["study_z"] = (study - study.mean()) / study.std()
analysis["gpa_band"] = pd.cut(analysis["final_gpa"], bins=[-0.01, 1, 2, 3, 4])

In [17]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [18]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv("northgate_clean.csv", index=False)

In [19]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing Values (study_hours_reported) -> Used central tendency (median) or class mean imputation to fill blank entries without distorting the overall workload distribution with extreme outliers.
- Housing Standardization (housing_clean) -> Cleaned whitespace, lowercased characters, and mapped raw string variations to consolidate text variations into three consistent groups (On-Campus, Off-Campus, and With Family).
- Impossible Values (impossible_ages & n_neg_work) -> Filtered out invalid ages ($\le 0$, $< 15$, or $> 100$) and negative work hours ($< 0$) caused by data entry errors.
- Duplicates (students_dedup) -> Removed exact/ID duplicate rows in student records using .drop_duplicates(subset=["student_id"]) to maintain a single record per student while leaving activity/enrollment event logs intact.
- Standardized Key (student_id_num) -> Stripped the "NU-" string prefix and cast the ID to an integer so student records could accurately join with the numeric sid key in course enrollments.
- Date Parsing (dates_parsed) -> Applied pd.to_datetime with format="mixed" to handle inconsistent text date formats across records without coercing valid entries into NaT.

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [20]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
clean_mean_gpa = analysis["final_gpa"].mean()
print(f"Clean Mean GPA: {clean_mean_gpa:.2f}")
corr_active_gpa = analysis["minutes_active"].corr(analysis["final_gpa"])
print(f"Correlation between active minutes and GPA: {corr_active_gpa:.2f}")

Clean Mean GPA: 2.31
Correlation between active minutes and GPA: 0.80


The mean GPA across all clean student records is 2.31.

There is a strong positive correlation (r = 0.80) between total LMS active minutes (minutes_active) and a student's final GPA, indicating that time spent engaged in course platforms is one of the strongest predictors of academic performance.

Cleaning duplicate records and standardizing student IDs allowed us to accurately merge the activity logs into a reliable dataset with one row per student without distorting core summary metrics.

In [21]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
